# Mapillary-Abdeckung — Osnabrück

Erfasst alle Mapillary-Bildstandorte im Stadtgebiet, wertet die Abdeckung aus und erzeugt einen reproduzierbaren, sequence-basierten Datensatz für die spätere VPR-Evaluation.

**Warum Vector Tiles und nicht die Bbox-Suche?**
`graph.mapillary.com/images?bbox=` ist eine Suchschnittstelle und antwortet
nachweislich unvollständig — gemessen wurde eine Kachel mit 30 Treffern, die
beim Vierteln 39 ergab. Es gibt keine Kachelgröße und keine Trefferzahl, an der
man das erkennen könnte, und es kommt weder Fehler noch Warnung.
Vector Tiles sind dieselbe Quelle, aus der mapillary.com seine Karte zeichnet:
vollständig per Konstruktion, und für Osnabrück reichen ~130 Abfragen.


`.env` im Projektordner mit `MAPILLARY_TOKEN=MLY|...` (Vorlage: `.env.example`).
Alle übrigen Parameter stehen in `config.yaml`.


**Datensatz-Spezifikation:** Die Coverage-Stufe erhält `image_id`, `sequence_id`, `captured_at`, `lat`, `lon`, `compass_angle`, `is_pano` und `creator_id`. Sequenzen werden als unteilbare Einheiten in `train`, `database` und `query` aufgeteilt. Für räumliche Beziehungen gelten 10 m als positiv, 10–25 m als unsicher und >25 m als negativ.

## 1. Konfiguration


In [ ]:
import json
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import mapbox_vector_tile
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
import yaml
from requests.adapters import HTTPAdapter, Retry
from shapely.geometry import Point


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


# Parameter: versioniert, damit alle im Team denselben Datensatz erzeugen.
cfg_file = find_upwards("config.yaml")
assert cfg_file, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(cfg_file.read_text())

CITY_NAME   = CFG["city"]
ZOOM        = CFG["zoom"]
MAX_WORKERS = CFG["max_workers"]



# Reproduzierbare MSLS-inspirierte Datensatz-Spezifikation.
# Sequenzen sind die Split-Einheit; einzelne Bilder werden niemals getrennt.
SPLIT_SEED = int(CFG.get("vpr", {}).get("split_seed", 42))
TRAIN_FRAC = float(CFG.get("vpr", {}).get("train_fraction", 0.70))
DATABASE_FRAC = float(CFG.get("vpr", {}).get("database_fraction", 0.15))
QUERY_FRAC = float(CFG.get("vpr", {}).get("query_fraction", 0.15))
POSITIVE_RADIUS_M = float(CFG.get("vpr", {}).get("positive_radius_m", 10.0))
UNCERTAIN_RADIUS_M = float(CFG["vpr"]["uncertain_radius_m"])
NEGATIVE_RADIUS_M = UNCERTAIN_RADIUS_M
MAX_HEADING_DIFF_DEG = float(CFG["vpr"]["max_heading_diff_deg"])


if MAX_WORKERS == "auto":
    MAX_WORKERS = min(16, (os.cpu_count() or 4) * 2)
else:
    MAX_WORKERS = int(MAX_WORKERS)

if not math.isclose(TRAIN_FRAC + DATABASE_FRAC + QUERY_FRAC, 1.0, rel_tol=0, abs_tol=1e-9):
    raise ValueError("vpr split fractions müssen zusammen 1.0 ergeben.")
if not (0 < POSITIVE_RADIUS_M < NEGATIVE_RADIUS_M):
    raise ValueError("vpr positive_radius_m muss > 0 und < uncertain_radius_m sein.")

# Auswertungen koennen unabhaengig voneinander ueber config.yaml
# aktiviert/deaktiviert werden. Standardmaessig bleiben beide aktiv.
DISTRICTS_ENABLED = CFG.get("districts", {}).get("enabled", True)
VERIFICATION_ENABLED = CFG.get("verification", {}).get("enabled", True)

# Token: geheim, deshalb bewusst nicht in config.yaml.
if env_file := find_upwards(".env"):
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

TOKEN = os.environ.get("MAPILLARY_TOKEN", "")
assert TOKEN.startswith("MLY|"), (
    "Kein Mapillary-Token. Datei .env anlegen:\n    MAPILLARY_TOKEN=MLY|dein|token"
)

PROJECT_ROOT = cfg_file.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "coverage"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = PROJECT_ROOT / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

F_TILES = PROCESSED_DIR / "vt_tiles.json"     # Rohpunkte je Kachel, dient als Checkpoint
F_ALL   = PROCESSED_DIR / "vt_images.csv"     # alle Punkte im Kachelbereich
F_CITY  = PROCESSED_DIR / "images_city.csv"   # Endergebnis: nur innerhalb der Stadtgrenze
F_METADATA = PROCESSED_DIR / "metadata.parquet"
F_TRAIN_SEQ = PROCESSED_DIR / "train_sequences.txt"
F_DATABASE_SEQ = PROCESSED_DIR / "database_sequences.txt"
F_QUERY_SEQ = PROCESSED_DIR / "query_sequences.txt"
F_POS_PAIRS = PROCESSED_DIR / "positive_candidates.csv"
F_NEG_PAIRS = PROCESSED_DIR / "negative_candidates.csv"
F_UNCERTAIN_PAIRS = PROCESSED_DIR / "uncertain_candidates.csv"

ox.settings.cache_folder = CACHE_DIR
ox.settings.use_cache = True
plt.rcParams["figure.dpi"] = 300

print(
    f"{CITY_NAME} | Zoom {ZOOM} | "
    f"Stadtteile {'✓' if DISTRICTS_ENABLED else '–'} | "
    f"Verifikation {'✓' if VERIFICATION_ENABLED else '–'} | "
    f"Konfig: {cfg_file.name} | Token ✓ |"
    f"CPU-Threads:     {os.cpu_count()} |"
    f"Download-Worker: {MAX_WORKERS}"
)


## 2. Hilfsfunktionen

Kachelmathematik nach der üblichen Web-Mercator-Konvention (wie OSM und Google
Maps), Dekodierung der binären Kachelformate und eine HTTP-Sitzung mit Retry.


In [ ]:
TILE_URL = "https://tiles.mapillary.com/maps/vtp/mly1_public/2/{z}/{x}/{y}"

# Die Dekodierbibliothek legt den Ursprung unten links, die Kachelspezifikation
# oben links. Gemessen gegen bekannte Koordinaten aus der Graph-API:
#   y_down=False -> 0,3 m Abweichung      y_down=True -> 699 m
# Rät man hier falsch, liegt alles an der Kachelmitte gespiegelt — und die Karte
# sieht trotzdem plausibel aus. Prüfzelle dazu im Anhang.
Y_DOWN = False

# Nur diese Felder liefern die Kacheln; sie reichen für die Abdeckungsanalyse.
# Kameratyp (Panorama, Fisheye) steht dort NICHT drin und muss später über die
# Graph-API geholt werden — für die paar tausend Bilder des Trainingssets.
PROPS = {
    "id": "image_id",
    "sequence_id": "sequence_id",
    "captured_at": "captured_at",
    "compass_angle": "compass_angle",
    "is_pano": "is_pano",
    "creator_id": "creator_id",
}


def make_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(
        max_retries=Retry(total=5, backoff_factor=0.6,
                          status_forcelist=[429, 500, 502, 503, 504],
                          allowed_methods=["GET"]),
        pool_maxsize=MAX_WORKERS * 4))
    return s


def deg2tile(lon, lat, z):
    n = 2 ** z
    return (int((lon + 180) / 360 * n),
            int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n))


def px2deg(tx, ty, px, py, extent, z):
    # Pixel innerhalb einer Kachel -> Lon/Lat
    n = 2 ** z
    if not Y_DOWN:
        py = extent - py
    wx, wy = (tx + px / extent) / n, (ty + py / extent) / n
    return (wx * 360 - 180,
            math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * wy)))))


def load_tile(session, tx, ty):
    r = session.get(TILE_URL.format(z=ZOOM, x=tx, y=ty),
                    params={"access_token": TOKEN}, timeout=60)
    r.raise_for_status()
    layer = mapbox_vector_tile.decode(r.content).get("image")
    if layer is None:
        return []
    extent = layer.get("extent", 4096)
    out = []
    for f in layer.get("features", []):
        geom = f.get("geometry") or {}
        if geom.get("type") != "Point":
            continue
        lon, lat = px2deg(tx, ty, *geom["coordinates"], extent, ZOOM)
        row = {"lon": lon, "lat": lat}
        row.update({PROPS[k]: v for k, v in (f.get("properties") or {}).items()
                    if k in PROPS})
        out.append(row)
    return out


## 3. Stadtgebiet und Kachelbereich


In [ ]:
# Stadtgrenze robust als Polygon bestimmen -- nicht einfach das erste
# Geocoder-Ergebnis uebernehmen, weil je nach Suchbegriff mehrere Objekte
# zurueckkommen koennen.
city_gdf = ox.geocode_to_gdf(CITY_NAME)
city_gdf = city_gdf[city_gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if city_gdf.empty:
    raise RuntimeError(f"Keine Polygon-Grenze fuer {CITY_NAME!r} gefunden.")

city_polygon = city_gdf.geometry.iloc[0]
UTM_CRS   = gpd.GeoSeries([city_polygon], crs="EPSG:4326").estimate_utm_crs()
city_area = gpd.GeoSeries([city_polygon], crs="EPSG:4326").to_crs(UTM_CRS).area.iloc[0] / 1e6

lon_min, lat_min, lon_max, lat_max = city_polygon.bounds
x0, y0 = deg2tile(lon_min, lat_max, ZOOM)     # obere linke Kachel
x1, y1 = deg2tile(lon_max, lat_min, ZOOM)     # untere rechte Kachel
TILES = [(x, y) for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)]

print(f"Fläche: {city_area:.1f} km²   Kacheln: {len(TILES)}")

## 4. Kacheln laden

Der Checkpoint wird alle zehn Kacheln geschrieben. Ein Abbruch kostet damit
höchstens ein paar Sekunden Arbeit; beim erneuten Ausführen wird nur geladen,
was noch fehlt.


In [ ]:
done = json.loads(F_TILES.read_text()) if F_TILES.exists() else {}
todo = [t for t in TILES if f"{t[0]}_{t[1]}" not in done]
print(f"{len(todo)} von {len(TILES)} Kacheln offen")


def fetch_tile(tile):
    # Eine Session pro Worker-Aufruf vermeidet offene Verbindungen und nutzt
    # trotzdem die Retry-/Connection-Pool-Konfiguration aus make_session().
    session = make_session()
    try:
        return tile, load_tile(session, *tile)
    finally:
        session.close()


failed_tiles = []

if todo:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        jobs = {pool.submit(fetch_tile, t): t for t in todo}
        for i, job in enumerate(as_completed(jobs), 1):
            try:
                (tx, ty), pts = job.result()
                done[f"{tx}_{ty}"] = pts
            except Exception as e:
                tile = jobs[job]
                failed_tiles.append(tile)
                print(f"\n  Kachel {tile} fehlgeschlagen: {type(e).__name__}: {e}")
            if i % 10 == 0 or i == len(todo):
                F_TILES.write_text(json.dumps(done))
                print(f"  {i}/{len(todo)} Kacheln — "
                      f"{sum(len(v) for v in done.values()):,} Punkte "
                      f"({time.time() - t0:.0f}s)", end="\r")
    F_TILES.write_text(json.dumps(done))

rows = [r for v in done.values() for r in v]
if not rows:
    raise RuntimeError("Keine Mapillary-Bilder aus den Vector Tiles geladen.")

images = pd.DataFrame(rows)
required = {"image_id", "sequence_id", "lon", "lat"}
missing = required - set(images.columns)
if missing:
    raise RuntimeError(
        f"Mapillary-Daten enthalten nicht alle benötigten Spalten: {sorted(missing)}"
    )

# Einheitliches Metadaten-Schema. Optional fehlende Tile-Properties bleiben als NA
# erhalten; dadurch bleibt die Datei auch bei unterschiedlichen Tile-Versionen stabil.
for col in ["captured_at", "compass_angle", "is_pano", "creator_id"]:
    if col not in images.columns:
        images[col] = pd.NA

images = images[
    ["image_id", "sequence_id", "captured_at", "lat", "lon",
     "compass_angle", "is_pano", "creator_id"]
].drop_duplicates(subset="image_id").reset_index(drop=True)

images.to_csv(F_ALL, index=False)

if failed_tiles:
    print(f"\n {len(failed_tiles)} Kacheln fehlgeschlagen. Sie werden beim nächsten Lauf erneut versucht.")

n_sequences = images["sequence_id"].nunique()
print(f"\n\n{len(images):,} Punkte, {n_sequences:,} Sequenzen -> {F_ALL}")

## 5. Auf das Stadtgebiet schneiden


In [ ]:
images_gdf = gpd.GeoDataFrame(
    images, geometry=gpd.points_from_xy(images["lon"], images["lat"]), crs="EPSG:4326")

# covered_by schliesst Punkte auf der Stadtgrenze ein. Das ist fuer GPS-Punkte
# sinnvoller als within, das Randpunkte ausschliesst.
images_gdf = images_gdf[images_gdf.geometry.covered_by(city_polygon)].reset_index(drop=True)
images_gdf.drop(columns="geometry").to_csv(F_CITY, index=False)

pct_city = len(images_gdf) / len(images) * 100 if len(images) else 0.0
print(f"Im Stadtgebiet: {len(images_gdf):,} von {len(images):,} ({pct_city:.1f} %)")
print(f"Dichte: {len(images_gdf) / city_area:,.0f} Bilder/km²   -> {F_CITY}")

## 6. Reproduzierbarer VPR-Datensatz

Die Coverage-Erhebung endet nicht bei einer reinen Bildzählung. Gemäß Spezifikation
werden die Metadaten als reproduzierbarer Datensatz gespeichert. Die Split-Einheit ist
`sequence_id`: Eine Sequenz liegt vollständig in genau einem von `train`, `database` oder
`query`.

Zusätzlich werden für die räumliche Evaluation GPS-Kandidaten erzeugt:
- **≤ 10 m:** positive candidate
- **10–25 m:** uncertain / nicht als sicher negativ verwenden
- **> 25 m:** negative candidate

Die Paare werden nur zwischen `query` und `database` erzeugt. Damit kann die spätere
Retrieval-Evaluation direkt auf denselben, unveränderten Splits aufbauen.


In [ ]:
# Sequence-aware Split und VPR-Metadaten.
# Wichtig: Wir splitten ausschließlich sequence_id, niemals einzelne Bilder.

import random

if images_gdf.empty:
    raise RuntimeError("Keine Bilder innerhalb der Stadtgrenze.")

metadata = images_gdf.drop(columns="geometry").copy()

# stabile, reproduzierbare Reihenfolge
seq_ids = sorted(metadata["sequence_id"].dropna().astype(str).unique())
if len(seq_ids) < 3:
    raise RuntimeError(
        f"Für train/database/query werden mindestens 3 Sequenzen benötigt; gefunden: {len(seq_ids)}"
    )

rng = random.Random(SPLIT_SEED)
rng.shuffle(seq_ids)

n = len(seq_ids)
n_train = max(1, round(n * TRAIN_FRAC))
n_database = max(1, round(n * DATABASE_FRAC))
# Query bekommt den Rest; dadurch summieren sich die gerundeten Anteile exakt.
if n_train + n_database >= n:
    n_database = max(1, n - n_train - 1)

train_sequences = seq_ids[:n_train]
database_sequences = seq_ids[n_train:n_train + n_database]
query_sequences = seq_ids[n_train + n_database:]

split_by_seq = (
    {s: "train" for s in train_sequences}
    | {s: "database" for s in database_sequences}
    | {s: "query" for s in query_sequences}
)
metadata["split"] = metadata["sequence_id"].astype(str).map(split_by_seq)

if metadata["split"].isna().any():
    raise RuntimeError("Mindestens eine Sequenz konnte keinem Split zugeordnet werden.")

# Reproduzierbare Artefakte
metadata.to_parquet(F_METADATA, index=False)
F_TRAIN_SEQ.write_text("\n".join(train_sequences) + "\n")
F_DATABASE_SEQ.write_text("\n".join(database_sequences) + "\n")
F_QUERY_SEQ.write_text("\n".join(query_sequences) + "\n")

print("Sequence-aware Split:")
print(f"  train:    {len(train_sequences):,} Sequenzen / {(metadata['split'] == 'train').sum():,} Bilder")
print(f"  database: {len(database_sequences):,} Sequenzen / {(metadata['split'] == 'database').sum():,} Bilder")
print(f"  query:    {len(query_sequences):,} Sequenzen / {(metadata['split'] == 'query').sum():,} Bilder")
print(f"  Seed:     {SPLIT_SEED}")
print(f"  -> {F_METADATA}")

# GPS-basierte Kandidatenbeziehungen. Für ~7.5k Bilder ist eine vollständige
# Distanzmatrix noch praktikabel, aber wir vermeiden eine N×N-Matrix und
# schreiben nur die tatsächlich relevanten Query→Database-Paare.
from scipy.spatial import cKDTree

db = metadata[metadata["split"] == "database"].copy()
qry = metadata[metadata["split"] == "query"].copy()

if db.empty or qry.empty:
    raise RuntimeError("Database und Query dürfen nicht leer sein.")

db_xy = gpd.GeoDataFrame(
    db, geometry=gpd.points_from_xy(db["lon"], db["lat"]), crs="EPSG:4326"
).to_crs(UTM_CRS)
qry_xy = gpd.GeoDataFrame(
    qry, geometry=gpd.points_from_xy(qry["lon"], qry["lat"]), crs="EPSG:4326"
).to_crs(UTM_CRS)

tree = cKDTree(np.c_[db_xy.geometry.x, db_xy.geometry.y])
qxy = np.c_[qry_xy.geometry.x, qry_xy.geometry.y]

db_x = db_xy.geometry.x.to_numpy()
db_y = db_xy.geometry.y.to_numpy()
db_pts = np.c_[db_x, db_y]
db_ids = db["image_id"].to_numpy()

# Alle DB-Bilder bis zum negativen Radius; >25m ist konzeptionell negativ,
# aber wir speichern nur die Kandidaten, die für die spätere Evaluation
# tatsächlich in einem definierten Nahbereich liegen. Zusätzlich wird die
# Anzahl der sicheren Negativen pro Query als Kennzahl ausgegeben.
near_lists = tree.query_ball_point(qxy, r=NEGATIVE_RADIUS_M)

positive_rows = []
uncertain_rows = []
negative_rows = []

for qi, db_idx_list in enumerate(near_lists):
    if not db_idx_list:
        continue
    qid = qry.iloc[qi]["image_id"]
    qpt = qxy[qi]
    for di in db_idx_list:
        
        dist_m = float(np.linalg.norm(qpt - db_pts[di]))
        row = {
            "query_image_id": qid,
            "database_image_id": db_ids[di],
            "distance_m": round(dist_m, 3),
        }
        if dist_m <= POSITIVE_RADIUS_M:
            positive_rows.append(row)
        else:
            uncertain_rows.append(row)

# Sichere Negative sind per Spezifikation >25m. Für den vollständigen Datensatz
# wird die Negativdefinition daher nicht künstlich auf einen 25m-Ring begrenzt.
# Statt einer riesigen Paardatei speichern wir pro Query repräsentative, weit
# entfernte Kandidaten deterministisch.
negative_rows = []
for qi, qrow in qry.reset_index(drop=True).iterrows():
    qpt = qxy[qi]
    distances = np.linalg.norm(db_pts - qpt, axis=1)
    valid = np.flatnonzero(distances > NEGATIVE_RADIUS_M)
    if len(valid):
        # Bis zu 50 sichere Negative pro Query, gleichmäßig über die DB verteilt.
        take = min(50, len(valid))
        positions = np.linspace(0, len(valid) - 1, take, dtype=int)
        for pos in positions:
            di = int(valid[pos])
            negative_rows.append({
                "query_image_id": qrow["image_id"],
                "database_image_id": db_ids[di],
                "distance_m": round(float(distances[di]), 3),
            })

pd.DataFrame(positive_rows, columns=["query_image_id", "database_image_id", "distance_m"]).to_csv(
    F_POS_PAIRS, index=False
)
pd.DataFrame(uncertain_rows, columns=["query_image_id", "database_image_id", "distance_m"]).to_csv(
    F_UNCERTAIN_PAIRS, index=False
)
pd.DataFrame(negative_rows, columns=["query_image_id", "database_image_id", "distance_m"]).to_csv(
    F_NEG_PAIRS, index=False
)

print("\nGPS-Kandidaten:")
print(f"  positive  ≤ {POSITIVE_RADIUS_M:g} m: {len(positive_rows):,} -> {F_POS_PAIRS}")
print(f"  uncertain {POSITIVE_RADIUS_M:g}–{NEGATIVE_RADIUS_M:g} m: {len(uncertain_rows):,} -> {F_UNCERTAIN_PAIRS}")
print(f"  negative  > {NEGATIVE_RADIUS_M:g} m: {len(negative_rows):,} repr. Paare -> {F_NEG_PAIRS}")

# Konsistenzprüfungen: Keine Sequenz darf in mehreren Splits vorkommen.
assert not (
    set(train_sequences) & set(database_sequences)
    or set(train_sequences) & set(query_sequences)
    or set(database_sequences) & set(query_sequences)
), "Sequence leakage im Split!"


In [ ]:
trn = metadata[metadata["split"] == "train"].copy()
trn_xy = gpd.GeoDataFrame(
    trn, geometry=gpd.points_from_xy(trn["lon"], trn["lat"]), crs="EPSG:4326"
).to_crs(UTM_CRS)

xy = np.c_[trn_xy.geometry.x, trn_xy.geometry.y]
seq = trn["sequence_id"].to_numpy()
ids = trn["image_id"].to_numpy()

tree_trn = cKDTree(xy)
pairs = tree_trn.query_pairs(r=POSITIVE_RADIUS_M, output_type="ndarray")

# ENTSCHEIDEND: Paare aus derselben Sequenz verwerfen (siehe unten).
mask = seq[pairs[:, 0]] != seq[pairs[:, 1]]
pairs = pairs[mask]

# --- Heading-Filter -------------------------------------------------
# Zyklische Winkeldifferenz: 350 Grad und 10 Grad sind 20 Grad auseinander.
ang = trn["compass_angle"].to_numpy(dtype=float)
dang = np.abs(ang[pairs[:, 0]] - ang[pairs[:, 1]])
dang = np.minimum(dang, 360.0 - dang)

# NaN behalten: fehlender compass_angle soll kein Paar verwerfen.
heading_ok = np.isnan(dang) | (dang <= MAX_HEADING_DIFF_DEG)

print(f"Paare nach Sequenzfilter:  {len(pairs):,}")
print(
    f"Paare nach Heading-Filter: {heading_ok.sum():,} ({heading_ok.mean() * 100:.1f} %)"
)
print(f"davon ohne compass_angle behalten: {int(np.isnan(dang).sum()):,}")

pairs = pairs[heading_ok]

dist = np.linalg.norm(xy[pairs[:, 0]] - xy[pairs[:, 1]], axis=1)
pd.DataFrame(
    {
        "anchor_image_id": ids[pairs[:, 0]],
        "positive_image_id": ids[pairs[:, 1]],
        "distance_m": dist.round(3),
    }
).to_csv(PROCESSED_DIR / "train_positive_candidates.csv", index=False)


## 7. Verifikation gegen mapillary.com

Feste Orte statt „dichtester Bereich": Letzterer zeigt per Konstruktion die
beste Stelle und taugt deshalb nicht als Kontrolle. Die ausgegebene URL öffnet
denselben Ausschnitt auf mapillary.com — direkt nebeneinanderlegbar.


In [ ]:
if not VERIFICATION_ENABLED:
    print("Übersprungen: Spot-Verifikation ist in config.yaml deaktiviert.")
else:
    verification_cfg = CFG.get("verification", {})
    SPOTS = {k: tuple(v) for k, v in verification_cfg.get("spots", {}).items()}
    RADIUS_M = verification_cfg.get("radius_m", 400)

    if not SPOTS:
        print("Keine Verifikations-Spots konfiguriert.")
    else:
        images_metric = images_gdf.to_crs(UTM_CRS)

        for name, (clat, clon) in SPOTS.items():
            center = gpd.GeoSeries(
                [Point(clon, clat)], crs="EPSG:4326"
            ).to_crs(UTM_CRS).iloc[0]
            sel_mask = images_metric.geometry.distance(center) <= RADIUS_M
            sel = images_gdf.loc[sel_mask].copy()

            fig, ax = plt.subplots(figsize=(7.5, 7.5))
            try:
                street_net = ox.graph_from_point(
                    (clat, clon), dist=RADIUS_M, network_type="all"
                )
                ox.plot_graph(
                    street_net, ax=ax, node_size=0, edge_linewidth=0.6,
                    edge_color="#cccccc", bgcolor="white",
                    show=False, close=False
                )
            except Exception as e:
                print(f"  (Straßennetz nicht geladen: {type(e).__name__})")

            dlat = RADIUS_M / 111_320
            dlon = RADIUS_M / (111_320 * np.cos(np.radians(clat)))
            ax.scatter(
                sel["lon"], sel["lat"], s=5, c="#2ca02c", alpha=0.5,
                edgecolors="none", zorder=5
            )
            ax.set_xlim(clon - dlon, clon + dlon)
            ax.set_ylim(clat - dlat, clat + dlat)
            ax.set_aspect(1 / np.cos(np.radians(clat)))
            ax.set_title(f"{name} — {len(sel):,} Bilder im {RADIUS_M}-m-Umkreis")
            plt.tight_layout()
            plt.show()

            n_sequences = (
                sel["sequence_id"].nunique()
                if "sequence_id" in sel.columns else 0
            )
            print(f"{name:<15} {len(sel):>7,} Bilder, {n_sequences:>4} Sequenzen")
            print(
                f"   https://www.mapillary.com/app/?lat={clat}&lng={clon}&z=17\n"
            )


## 8. Stadtweite Karte


In [ ]:
# network_type="drive" reicht als Hintergrund und lädt deutlich schneller
# als "all" (das zusätzlich Fuß- und Radwege enthält).
street_net = ox.graph_from_polygon(city_polygon, network_type="drive")

fig, ax = ox.plot_graph(street_net, node_size=0, edge_linewidth=0.35,
                        edge_color="#cccccc", bgcolor="white",
                        show=False, close=False, figsize=(14, 14))
ax.scatter(images_gdf["lon"], images_gdf["lat"], s=0.8, c="#2ca02c",
           alpha=0.35, edgecolors="none", zorder=5)
gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
    ax=ax, color="black", linewidth=1.2, linestyle="--")
ax.set_title(f"{CITY_NAME.split(',')[0]} — {len(images_gdf):,} Mapillary-Bilder")
plt.tight_layout(); plt.savefig(PROCESSED_DIR / "coverage_city.png"); plt.show()


## 9. Stadtteile bestimmen

Die gesamte Stadtteilberechnung kann in `config.yaml` mit `districts.enabled` aktiviert oder deaktiviert werden.

Die administrative Gliederung ist die Primaerquelle. Zuerst werden `boundary=administrative`-Flaechen innerhalb der bereits bekannten Stadtgrenze geladen und nur die konfigurierten `admin_level`-Werte betrachtet. Standardmaessig wird `admin_level=10` vor `admin_level=9` bevorzugt.

Die Auswahl prueft:
1. **Tatsaechliche Coverage** — Union der Polygone statt einfacher Flächensumme.
2. **Ueberlappung** — doppelt bedeckte Flaechen werden als Qualitaetsproblem erkannt.
3. **Mindestanzahl und Mindestflaeche** — sehr kleine oder unbrauchbare Kandidaten werden entfernt.
4. **Clippen an der Stadtgrenze** — die zurueckgegebenen Polygone liegen wirklich im Stadtgebiet.
5. **place-Tags als Fallback** — nur wenn keine brauchbare administrative Ebene vorhanden ist.

Damit werden unterschiedliche Hierarchieebenen nicht mehr anschliessend per Geometrieheuristik vermischt.

In [ ]:
DISTRICTS = CFG["districts"]
PLACE_TAGS       = DISTRICTS["place_tags"]
MIN_AREA_KM2     = DISTRICTS["min_area_km2"]
ADMIN_LEVELS     = DISTRICTS.get("admin_levels", ["10", "9"])
MIN_DISTRICTS_REQ = DISTRICTS.get("min_districts_required", 3)


def get_districts_by_place_tag(city_name, tags):
    """Fallback fuer Staedte, deren Stadtteile nicht als administrative Flaechen erfasst sind."""
    try:
        gdf = ox.features_from_place(city_name, tags={"place": tags})
        gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
        if "name" not in gdf.columns:
            return None
        gdf = gdf[gdf["name"].notna()][["name", "geometry"]].reset_index(drop=True)
        return gdf if len(gdf) > 0 else None
    except Exception as e:
        print(f"Fallback (place-Tags) fehlgeschlagen: {e}")
        return None


def get_admin_districts(city_polygon, admin_levels):
    """Holt administrative Grenzen innerhalb der bereits bekannten Stadtgrenze."""
    try:
        gdf = ox.features_from_polygon(city_polygon, tags={"boundary": "administrative"})
        gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
        if "name" not in gdf.columns or "admin_level" not in gdf.columns:
            return None
        gdf = gdf[gdf["name"].notna() & gdf["admin_level"].notna()].copy()
        gdf["admin_level"] = gdf["admin_level"].astype(str)
        gdf = gdf[gdf["admin_level"].isin([str(x) for x in admin_levels])]
        return gdf.reset_index(drop=True) if len(gdf) > 0 else None
    except Exception as e:
        print(f"Strategie A (admin_level) fehlgeschlagen: {e}")
        return None


def diagnose_osm_districts(city_name):
    """Zeigt, welche Stadtteil-Schemata OSM fuer diese Stadt ueberhaupt kennt."""
    print("\n" + "=" * 62)
    print(f"DIAGNOSE: Was kennt OSM fuer '{city_name}'?")
    print("=" * 62)
    for tag_key, tag_vals in [("place", ["suburb", "quarter", "neighbourhood",
                                            "borough", "village", "hamlet"]),
                              ("boundary", ["administrative"])]:
        try:
            gdf = ox.features_from_polygon(city_polygon, tags={tag_key: tag_vals})
            if len(gdf) == 0:
                print(f"\n{tag_key}: keine Treffer")
                continue
            print(f"\n{tag_key}: {len(gdf)} Objekte")
            for gtype, cnt in gdf.geometry.type.value_counts().items():
                mark = " <- nutzbar" if gtype in ("Polygon", "MultiPolygon") else " (keine Flaeche)"
                print(f"  {gtype:15s} {cnt:4d}{mark}")
            if tag_key == "place" and "place" in gdf.columns:
                print(f"  Werte: {dict(gdf['place'].value_counts())}")
            if tag_key == "boundary" and "admin_level" in gdf.columns:
                poly = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]
                if len(poly):
                    print(f"  admin_level (nur Flaechen): "
                          f"{dict(poly['admin_level'].value_counts().sort_index())}")
        except Exception as e:
            print(f"\n{tag_key}: Abfrage fehlgeschlagen ({e})")
    print("\n" + "-" * 62)
    print("Interpretation:")
    print(" - Nur 'Point' -> Stadtteile sind nur als Punkte getaggt, nicht als Flaechen.")
    print(" - admin_level 10/9 werden bevorzugt; andere Ebenen sind nicht automatisch Stadtteile.")
    print(" - place=* dient nur als Fallback, wenn keine brauchbare administrative Ebene existiert.")
    print("=" * 62 + "\n")


def prepare_districts(gdf, city_polygon, target_crs):
    """Repariert, clippt und berechnet Flaechen erst nach dem Clippen."""
    if gdf is None or gdf.empty:
        return None

    gdf = gdf.copy()
    n_invalid = int((~gdf.geometry.is_valid).sum())
    if n_invalid:
        print(f"{n_invalid} ungueltige Polygone repariert (buffer(0))")
        gdf.loc[~gdf.geometry.is_valid, "geometry"] = (
            gdf.loc[~gdf.geometry.is_valid, "geometry"].buffer(0)
        )

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf["geometry"] = gdf.geometry.intersection(city_polygon)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

    metric = gdf.to_crs(target_crs)
    gdf["area_km2"] = metric.area.values / 1e6
    gdf = gdf[gdf["area_km2"] >= MIN_AREA_KM2].copy()

    # Gleiche Namen koennen in OSM mehrfach vorkommen. Fuer die Bildzuordnung
    # bleibt je Name die groessere Flaeche erhalten.
    gdf = (gdf.sort_values("area_km2", ascending=False)
             .drop_duplicates(subset="name", keep="first")
             .reset_index(drop=True))
    return gdf if len(gdf) else None


def level_quality(gdf, city_polygon, city_area, target_crs):
    """Bewertet Coverage und echte Ueberlappung eines admin_levels."""
    metric = gdf.to_crs(target_crs)
    area_sum = metric.area.sum() / 1e6
    union_geom = metric.geometry.union_all()
    union_area = gpd.GeoSeries([union_geom], crs=target_crs).area.iloc[0] / 1e6
    coverage = union_area / city_area if city_area > 0 else 0.0
    overlap = 1.0 - union_area / area_sum if area_sum > 0 else 0.0
    return area_sum, coverage, max(0.0, overlap)



if not DISTRICTS_ENABLED:
    districts = None
    districts_gdf = None
    HAS_DISTRICTS = False
    print("Stadtteil-Auswertung deaktiviert (districts.enabled: false).")
else:
    print(f"Lade Stadtteile fuer: {CITY_NAME}")

    # Administrative Grenzen sind die Primaerquelle. admin_level 10 wird vor 9
    # bevorzugt, sofern die konfigurierte Reihenfolge dies vorsieht.
    admin_candidates = get_admin_districts(city_polygon, ADMIN_LEVELS)

    best_level = None
    best_gdf = None
    best_score = float("-inf")
    level_stats = {}

    if admin_candidates is not None:
        for level in ADMIN_LEVELS:
            level = str(level)
            level_gdf = admin_candidates[admin_candidates["admin_level"] == level].copy()
            level_gdf = prepare_districts(level_gdf, city_polygon, UTM_CRS)
            if level_gdf is None or len(level_gdf) < MIN_DISTRICTS_REQ:
                continue

            area_sum, coverage, overlap = level_quality(level_gdf, city_polygon, city_area, UTM_CRS)
            coverage_score = 1.0 - abs(1.0 - coverage)
            score = coverage_score - overlap
            level_stats[level] = {
                "count": len(level_gdf),
                "coverage": coverage,
                "overlap": overlap,
                "score": score,
            }

            # Bei vergleichbarer Qualitaet gewinnt die konfigurierte Reihenfolge.
            if score > best_score:
                best_score = score
                best_level = level
                best_gdf = level_gdf

    if level_stats:
        print("\nAnalysierte admin_levels:")
        for level in ADMIN_LEVELS:
            level = str(level)
            if level not in level_stats:
                continue
            stats = level_stats[level]
            print(f" - Level {level:>2}: {stats['count']:3d} Polygone, "
                  f"Coverage {stats['coverage'] * 100:6.1f}%, "
                  f"Overlap {stats['overlap'] * 100:5.1f}%")

    if best_gdf is not None:
        districts = best_gdf
        print(f"\n-> Verwende admin_level={best_level} als Stadtteilgliederung.")
    else:
        print("\nKeine ausreichend gute administrative Ebene gefunden.")
        print("-> Versuche place-Tags als Fallback...")
        districts = prepare_districts(
            get_districts_by_place_tag(CITY_NAME, PLACE_TAGS),
            city_polygon,
            UTM_CRS,
        )

    HAS_DISTRICTS = districts is not None and len(districts) >= MIN_DISTRICTS_REQ
    districts_gdf = None

    if not HAS_DISTRICTS:
        print("\nKeine ausreichenden Stadtteil-Flaechen in OSM gefunden.")
        diagnose_osm_districts(CITY_NAME)
    else:
        districts_gdf = gpd.GeoDataFrame(districts, geometry="geometry", crs="EPSG:4326")
        print(f"\n{len(districts_gdf)} Stadtteile final:")
        for _, row in districts_gdf.sort_values("name").iterrows():
            print(f"  {row['name']:32s} {row['area_km2']:6.2f} km²")

        # Echte Abdeckung = Union der Stadtteilflaechen, nicht Summe der Einzelbereiche.
        metric = districts_gdf.to_crs(UTM_CRS)
        union_area = metric.geometry.union_all().area / 1e6
        coverage_pct = union_area / city_area * 100 if city_area > 0 else float("nan")
        print(f"\nTatsaechliche Stadtteilabdeckung: {union_area:.1f} km² ({coverage_pct:.0f}% der Stadtflaeche)")
        if coverage_pct < 80:
            print("Unter 80% -- Luecken zwischen den Polygonen. Bilder dort bleiben unzugeordnet.")

    if not HAS_DISTRICTS:
        print("=" * 62)
        print("KEINE STADTTEILE VERFUEGBAR")
        print("Uebersprungen werden: Metriken pro Stadtteil, Choroplethen-Karte.")
        print("Weiterhin ausgewertet: stadtweite Heatmap, Strassennetz, Gesamtmetriken.")
        print("=" * 62)

## 10. Abdeckung pro Stadtteil


In [ ]:
if not HAS_DISTRICTS:
    joined, metrics_df = None, None
    print("Uebersprungen: zu wenige Stadtteile gefunden.")
else:
    # covered_by schliesst Grenzpunkte ein. Falls ein Punkt mehrere Treffer hat,
    # wird nicht zufaellig der erste Datensatz genommen, sondern die kleinste
    # passende Flaeche. Das ist deterministisch und bevorzugt die feinere Einheit.
    right = districts_gdf[["name", "area_km2", "geometry"]].rename(
        columns={"name": "district", "area_km2": "district_area_km2"})
    joined = gpd.sjoin(images_gdf, right, how="left", predicate="covered_by")

    n_dupes = int(joined["image_id"].duplicated().sum())
    if n_dupes:
        print(f"{n_dupes} Bilder mehrfach zugeordnet -- kleinste passende Flaeche wird behalten")
        joined = (joined.sort_values(["image_id", "district_area_km2"],
                                     na_position="last")
                        .drop_duplicates(subset="image_id", keep="first")
                        .reset_index(drop=True))

    n_unassigned = int(joined["district"].isna().sum())
    pct_unassigned = n_unassigned / len(joined) * 100 if len(joined) else 0.0
    print(f"Ohne Stadtteil-Zuordnung: {n_unassigned:,} ({pct_unassigned:.1f}%)")
    if pct_unassigned > 20:
        print("Ueber 20% unzugeordnet -- die Stadtteil-Polygone decken das Stadtgebiet "
              "nur lueckenhaft ab.")
    print()

    metrics_rows = []
    for _, r in districts_gdf.iterrows():
        mask = joined["district"] == r["name"]
        n_images = int(mask.sum())
        n_sequences = joined.loc[mask, "sequence_id"].nunique() if "sequence_id" in joined.columns else 0
        density = n_images / r["area_km2"] if r["area_km2"] > 0 else float("nan")
        metrics_rows.append({
            "Stadtteil": r["name"],
            "Fläche km²": round(r["area_km2"], 2),
            "Bilder": n_images,
            "Bilder/km²": round(density),
            "Sequenzen": n_sequences,
        })

    metrics_df = (pd.DataFrame(metrics_rows)
                  .sort_values("Bilder/km²", ascending=False)
                  .reset_index(drop=True))

    zugeordnet = int(metrics_df["Bilder"].sum())
    median = metrics_df["Bilder/km²"].median()
    pct_assigned = zugeordnet / len(images_gdf) * 100 if len(images_gdf) else 0.0
    print(f"Zugeordnet: {zugeordnet:,} von {len(images_gdf):,} ({pct_assigned:.1f} %)")
    print(f"Median:     {median:,.0f} Bilder/km²\n")

    schwach = metrics_df[metrics_df["Bilder/km²"] < median * 0.5]
    if len(schwach):
        print(f"{len(schwach)} Stadtteile unter der Haelfte des Medians:")
        for _, r in schwach.iterrows():
            print(f"   {r['Stadtteil']:<32} {r['Bilder/km²']:>8,.0f}")

metrics_df

## 11. Choroplethenkarte


In [ ]:
if not HAS_DISTRICTS:
    print("Übersprungen: braucht Stadtteile.")
else:
    plot_gdf = districts_gdf.merge(metrics_df, left_on="name",
                                   right_on="Stadtteil", how="left")

    fig, ax = plt.subplots(figsize=(13, 11))
    plot_gdf.plot(column="Bilder/km²", cmap="YlOrRd", legend=True, ax=ax,
                  edgecolor="black", linewidth=0.5,
                  legend_kwds={"label": "Bilder pro km²", "shrink": 0.7},
                  missing_kwds={"color": "lightgrey", "label": "keine Daten"})

    for _, r in plot_gdf.iterrows():
        p = r.geometry.representative_point()      # liegt garantiert im Polygon
        ax.annotate(r["name"], xy=(p.x, p.y), fontsize=7, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.65, ec="none"))

    gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
        ax=ax, color="black", linewidth=1.5, linestyle="--")
    ax.set_aspect(1 / np.cos(np.radians(city_polygon.centroid.y)))
    ax.set_title(f"Mapillary-Abdeckung nach Stadtteil — {CITY_NAME.split(',')[0]}")
    plt.tight_layout(); plt.savefig(FIGURE_DIR / "choropleth_coverage.png"); plt.show()


## 12. Dichte-Heatmap

Geglättetes 2D-Histogramm statt Kerndichteschätzung: eine echte KDE müsste für
jeden Rasterpunkt über alle 300.000 Bilder summieren und läuft bei dieser Menge
praktisch nicht mehr. Das Ergebnis sieht gleich aus und rechnet in Sekunden.


In [ ]:
from scipy.ndimage import gaussian_filter

pad_x = (lon_max - lon_min) * 0.03
pad_y = (lat_max - lat_min) * 0.03
extent = [lon_min - pad_x, lon_max + pad_x, lat_min - pad_y, lat_max + pad_y]

density, _, _ = np.histogram2d(images_gdf["lon"], images_gdf["lat"],
                               bins=[420, 300],
                               range=[extent[:2], extent[2:]])
density = gaussian_filter(density, sigma=2.0)

fig, ax = plt.subplots(figsize=(13, 11))
# log1p, weil die Innenstadt sonst alles andere überstrahlt
im = ax.imshow(np.log1p(density).T, origin="lower", extent=extent,
               cmap="inferno", interpolation="bilinear", aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.7, label="Bilddichte (logarithmisch)")

gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
    ax=ax, color="white", linewidth=1.5, linestyle="--")
ax.set_aspect(1 / np.cos(np.radians(city_polygon.centroid.y)))
ax.set_title(f"Bilddichte — {CITY_NAME.split(',')[0]} ({len(images_gdf):,} Bilder)")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "heatmap_coverage.png"); plt.show()


## 13. Bericht


In [ ]:
per_seq = images_gdf["sequence_id"].value_counts() if "sequence_id" in images_gdf.columns else pd.Series(dtype="int64")
captured = pd.to_datetime(images_gdf["captured_at"], unit="ms", errors="coerce")
valid_captured = captured.dropna()

print("=" * 58)
print(f"  Stadt                {CITY_NAME}")
print(f"  Fläche               {city_area:,.1f} km²")
print(f"  Bilder               {len(images_gdf):,}")
print(f"  Dichte               {len(images_gdf) / city_area:,.0f} /km²")
print(f"  Sequenzen            {per_seq.size:,}")
if len(per_seq):
    print(f"  Median je Sequenz    {per_seq.median():,.0f}   (längste: {per_seq.max():,})")
if len(valid_captured):
    print(f"  Aufnahmezeitraum     {valid_captured.min():%Y-%m} bis {valid_captured.max():%Y-%m}")
else:
    print("  Aufnahmezeitraum     nicht verfügbar")
if HAS_DISTRICTS:
    print(f"  Stadtteile           {len(districts_gdf)}")
print("=" * 58)

print(f"  Split                train={len(train_sequences):,} / database={len(database_sequences):,} / query={len(query_sequences):,}")
print(f"  Positive-Radius      ≤ {POSITIVE_RADIUS_M:g} m")
print(f"  Negative-Radius      > {NEGATIVE_RADIUS_M:g} m")
print(f"  Metadaten            {F_METADATA}")


## Anhang — Achsenkalibrierung

Nur nötig, wenn sich die Dekodierbibliothek ändert oder die Punkte plötzlich
verschoben aussehen. Vergleicht beide Achsenrichtungen gegen Koordinaten aus
der Graph-API und meldet, welche stimmt. Braucht eine Referenzdatei mit den
Spalten `image_id`, `lon`, `lat`.


In [ ]:
REF_FILE = PROCESSED_DIR / "images_raw.csv"      # bei Bedarf anpassen

if not REF_FILE.exists():
    print(f"Keine Referenzdatei ({REF_FILE}) — Kalibrierung übersprungen.")
else:
    ref = pd.read_csv(REF_FILE).set_index("image_id")[["lon", "lat"]]
    cx, cy = deg2tile((lon_min + lon_max) / 2, (lat_min + lat_max) / 2, ZOOM)
    session = make_session()
    original_y_down = Y_DOWN

    try:
        for candidate in (False, True):
            Y_DOWN = candidate
            pts = pd.DataFrame(load_tile(session, cx, cy))
            both = pts.set_index("image_id").join(ref, rsuffix="_ref", how="inner").dropna()
            if both.empty:
                print(f"  y_down={candidate}: keine gemeinsamen Bilder")
                continue
            dist = np.hypot(
                (both["lon"] - both["lon_ref"]) * 111_320 * np.cos(np.radians(both["lat"])),
                (both["lat"] - both["lat_ref"]) * 111_320)
            print(f"  y_down={candidate!s:<5} {len(both):>5} Treffer, "
                  f"Medianabweichung {dist.median():>10,.1f} m")
    finally:
        Y_DOWN = original_y_down
        session.close()

    print("\nDer kleinere Wert gewinnt. Liegen beide über 25 m, stimmt an der "
          "Dekodierung etwas nicht.")